In [1]:
import numpy as np
import tensorflow as tf
import re
import pickle
import time
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import multiprocessing as mp


2026-03-07 16:08:28.005590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772899708.495688      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772899708.624438      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772899709.661456      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772899709.661490      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772899709.661492      24 computation_placer.cc:177] computation placer alr

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# Disable mixed precision for stability (float32 is safer)
tf.keras.mixed_precision.set_global_policy('float32')
print("GPUs:", tf.config.list_physical_devices('GPU'))


GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [4]:
#  Multi‑GPU Strategy
strategy = tf.distribute.MirroredStrategy()
print("Number of GPUs:", strategy.num_replicas_in_sync)
GLOBAL_BATCH_SIZE = 256          # must be divisible by number of GPUs
assert GLOBAL_BATCH_SIZE % strategy.num_replicas_in_sync == 0


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of GPUs: 2


I0000 00:00:1772899751.579719      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772899751.586073      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:

#  Load Dataset (70,000 stories)


dataset = load_dataset("roneneldan/TinyStories", split="train")
num_stories = 70000
dataset = dataset.select(range(num_stories))
print(f"Using {num_stories} stories")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Using 70000 stories


In [6]:
#  Text Preprocessing with Stopwords & Lemmatization
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

print("Cleaning texts with multiprocessing...")
start_clean = time.time()
raw_texts = [x["text"] for x in dataset]

with mp.Pool(processes=mp.cpu_count()) as pool:
    texts = pool.map(clean_text, raw_texts)

# Remove empty stories (all stopwords, etc.)
texts = [t for t in texts if len(t) > 0]
print(f"Remaining stories after cleaning: {len(texts)}")
print(f"Preprocessing took {time.time() - start_clean:.2f} seconds")


Cleaning texts with multiprocessing...
Remaining stories after cleaning: 69988
Preprocessing took 17.22 seconds


In [7]:
#  Tokenization



vocab_size = 10000
seq_length = 30

tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>",
    filters=''
)
tokenizer.fit_on_texts(texts)

# total_words = actual vocab size + 1 (for padding index 0)
total_words = len(tokenizer.word_index) + 1
print("Vocabulary size (embedding input_dim):", total_words)

sequences = tokenizer.texts_to_sequences(texts)
print("Number of stories after tokenization:", len(sequences))

Vocabulary size (embedding input_dim): 14279
Number of stories after tokenization: 69988


In [8]:
# 6. Create n‑gram Sequences (input, target)

input_sequences = []
target_words = []

print("Creating n‑grams...")
start_ngrams = time.time()
for seq in sequences:
    if len(seq) <= seq_length:
        continue
    for i in range(seq_length, len(seq)):
        input_sequences.append(seq[i-seq_length:i])
        target_words.append(seq[i])
print(f"Created {len(input_sequences)} n‑grams in {time.time() - start_ngrams:.2f} seconds")

X = np.array(input_sequences)
y = np.array(target_words)

# Ensure no label 0 (padding index) – safety check
if np.any(y == 0):
    print("WARNING: Found label 0 – filtering those samples.")
    mask = y != 0
    X = X[mask]
    y = y[mask]
    print(f"After filtering: X shape {X.shape}, y shape {y.shape}")

assert np.all(y >= 1) and np.all(y < total_words), "Labels out of range!"
print("X shape:", X.shape)

Creating n‑grams...
Created 4036967 n‑grams in 7.42 seconds
X shape: (4036967, 30)


In [9]:
# 7. Build tf.data Pipeline (with drop_remainder=True)

# Important: drop_remainder ensures all batches have exactly GLOBAL_BATCH_SIZE,
# eliminating the shape mismatch across GPUs.
dataset_tf = tf.data.Dataset.from_tensor_slices((X, y))
dataset_tf = dataset_tf.shuffle(100000)
dataset_tf = dataset_tf.batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
dataset_tf = dataset_tf.prefetch(tf.data.AUTOTUNE)
dataset_tf = dataset_tf.cache()   # optional, speeds up after first epoch

# Verify that the dataset has at least one batch
num_batches = len(X) // GLOBAL_BATCH_SIZE
print(f"Number of full batches: {num_batches}")


Number of full batches: 15769


In [10]:
# 8. Build Model inside strategy scope
embedding_dim = 128
lstm_units = 256
learning_rate = 3e-4   # low LR for stability

with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(total_words, embedding_dim, input_length=seq_length),
        tf.keras.layers.LSTM(lstm_units, return_sequences=False),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(total_words, activation='softmax')
    ])

    # Build explicitly (optional but good)
    model.build(input_shape=(None, seq_length))

    # Use gradient clipping by value to prevent NaN loss
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, clipvalue=0.5)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 128)        │     1,827,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14279)          │     3,669,703 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,957,447 (22.73 MB)

 Trainable params: 5,957,447 (22.73 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 9. Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='loss', patience=2, restore_best_weights=True
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', factor=0.5, patience=1, min_lr=1e-6
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras', monitor='loss', save_best_only=True
)

In [12]:
# 10. Train Model


epochs = 15
print("Starting training...")
start_train = time.time()
history = model.fit(
    dataset_tf,          # Keras automatically distributes this dataset with MirroredStrategy
    epochs=epochs,
    callbacks=[early_stop, lr_scheduler, checkpoint]
)
print(f"Training took {time.time() - start_train:.2f} seconds")


Starting training...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

I0000 00:00:1772899812.552599      66 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772899812.663688      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


15767/15769 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0431 - loss: 6.6462INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 329s 20ms/step - accuracy: 0.0431 - loss: 6.6461 - learning_rate: 3.0000e-04
Epoch 2/15
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 319s 20ms/step - accuracy: 0.1030 - loss: 5.7450 - learning_rate: 3.0000e-04
Epoch 3/15
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 319s 20ms/step - accuracy: 0.1223 - loss: 5.4511 - learning_rate: 3.0000e-04
Epoch 4/15
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 320s 20ms/step - accuracy: 0.1328 - loss: 5.2923 - learning_rate: 3.0000e-04
Epoch 5/15
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 324s 20ms/step - accuracy: 0.1401 - loss: 5.1882 - learning_rate: 3.0000e-04
Epoch 6/15
15769/15769 ━━━━━━━━━━━━━━━━━━━━ 326

In [13]:
# 11. Save Model & Tokenizer

model.save("tinystories_lstm_70k_final.keras")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("Model and tokenizer saved successfully!")


Model and tokenizer saved successfully!


In [14]:
# 12. Prediction Helper

def predict_top_k(model, tokenizer, text, seq_length=30, k=5, temperature=0.8):
    text = text.lower()
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = tf.keras.preprocessing.sequence.pad_sequences(
        [token_list], maxlen=seq_length, padding="pre"
    )
    preds = model.predict(token_list, verbose=0)[0]
    preds = np.log(preds + 1e-9) / temperature
    preds = np.exp(preds)
    preds = preds / np.sum(preds)
    top_idx = preds.argsort()[-k:][::-1]
    results = [(tokenizer.index_word.get(idx, ""), float(preds[idx])) for idx in top_idx]
    return results